# LSTM load forecasting

Generated notebook for the GridMind IS2 project. Run from the project root or use the first cell to configure the path.

In [ ]:
from pathlib import Path
import sys
from importlib import import_module
ROOT=Path.cwd()
if not (ROOT/'config.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
source=import_module('01_data_acquisition.data_source'); prep=import_module('02_data_preprocessing.preprocess'); lstm=import_module('03_load_forecasting.lstm_forecaster')

In [ ]:
raw=source.generate_building_data(days=90,seed=42)
clean=prep.clean_and_engineer(raw)
train,val,test=prep.chronological_split(clean)
print(len(train),len(val),len(test))

 Train-only normalization is performed inside train_lstm. Start with a small epoch count.
bundle=lstm.train_lstm(train,val,epochs=8,lookback=24,horizon=24,seed=42)
bundle['history'][-3:]

In [ ]:
actual,pred=lstm.predict_windows(bundle,test)
metrics=lstm.forecast_metrics(actual,pred)
metrics

In [ ]:
import pandas as pd
import plotly.graph_objects as go
fig=go.Figure(); fig.add_trace(go.Scatter(y=actual[0],name='Actual')); fig.add_trace(go.Scatter(y=pred[0],name='LSTM prediction'))
fig.update_layout(title='First 24-step held-out forecast',xaxis_title='Forecast step',yaxis_title='Load (kW)')
fig.show()

 Empirical uncertainty interval from validation residuals; approximate, not Bayesian.
from importlib import import_module
solar=import_module('04_solar_forecasting.solar')
residuals=(actual-pred).reshape(-1)
lo,hi=solar.empirical_interval(pred[0],residuals,coverage=.90)
{'approx_90_interval_first_steps':list(zip(lo[:5].round(2),hi[:5].round(2)))}